# Module 1: Environment Mastery & Path Architecture

Welcome to Phase 2. As developers transition from writing isolated scripts to building complex, production-ready systems, the most common roadblock isn't the code itself—it's the environment. 

In this module, we will explore the mechanics of isolation. We will move beyond simply running Python to understanding the underlying operating system linkages, dependency management, and dynamic path routing required for scalable architecture.

Run the cell below to identify the exact Python binary executing this notebook.

In [2]:
import sys
import site

print("=== 1. Execution Environment Diagnostics ===")
# Identifies the exact binary running this process. 
# This is crucial for ensuring you aren't accidentally running the system Python.
print(f"Active Python Binary: {sys.executable}")
print(f"Python Version: {sys.version.split(' ')[0]}\n")

# Determine the exact location where pip/conda is installing third-party packages
print("=== Package Installation Paths ===")
print(f"Global Site-Packages:\n{site.getsitepackages()}\n")
print(f"User Site-Packages:\n{site.getusersitepackages()}")

=== 1. Execution Environment Diagnostics ===
Active Python Binary: c:\Users\zarya\anaconda3\python.exe
Python Version: 3.11.3

=== Package Installation Paths ===
Global Site-Packages:
['c:\\Users\\zarya\\anaconda3', 'c:\\Users\\zarya\\anaconda3\\Lib\\site-packages']

User Site-Packages:
C:\Users\zarya\AppData\Roaming\Python\Python311\site-packages


## The Necessity of Virtual Environments

Global installations create state mutations. If you upgrade a package globally for a lightweight web API, you might silently break a YOLO benchmarking script or a 3D segmentation model in another project because they relied on the older version.

**The Solutions:**
1. **`venv` (Standard Library):** Creates a lightweight directory with symlinks to the system's Python binary. Best for standard API wrappers and microservices.
2. **`conda`:** A cross-language package manager that handles underlying C/C++ libraries and binaries. Essential for AI workloads requiring specific CUDA toolkits or OpenCV binaries.

The code below programmatically verifies that we are operating safely inside an isolated "bubble."

In [3]:
import os

print("=== 2. Strict Virtual Environment Validation ===")

def verify_environment():
    """Validates if the script is running in an isolated environment."""
    # Check for standard venv
    is_venv = hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)
    
    # Check for Conda (looks for the conda-meta folder in the environment path)
    is_conda = os.path.exists(os.path.join(sys.prefix, 'conda-meta'))
    
    if is_conda:
        print("Status: Conda Environment DETECTED.")
        print("Excellent for handling heavy C++ binaries and CUDA dependencies for Vision tasks.")
    elif is_venv:
        print("Status: Standard venv DETECTED.")
        print("Good for lightweight Python modules and general engineering tasks.")
    else:
        print("WARNING: Running on Global System Python.")
        print("This violates production isolation standards. Please activate an environment.")

verify_environment()

=== 2. Strict Virtual Environment Validation ===
Status: Conda Environment DETECTED.
Excellent for handling heavy C++ binaries and CUDA dependencies for Vision tasks.


## Python's Internal Routing (`sys.path`)

Once the Operating System finds Python, how does Python find your custom code? When you type `import my_module`, Python searches directories in a strict order dictated by `sys.path`.

If you have a complex repository structure (e.g., separating `notebooks/`, `src/`, and `tests/`), standard relative imports will fail. To fix this without relying on fragile hacks, professional pipelines dynamically alter `sys.path` at runtime.

In [4]:
print("=== 3. Dynamic Path Resolution ===")

# 1. Get the absolute path of the current working directory
current_dir = os.path.abspath(os.getcwd())
print(f"Current Directory: {current_dir}")

# 2. Programmatically find the parent directory (project root)
project_root = os.path.dirname(current_dir)

# 3. Define the path to our custom source code (assuming a 'src' folder exists)
custom_src_path = os.path.join(project_root, 'src')

# 4. Inject it safely into Python's routing
if custom_src_path not in sys.path:
    # Insert at index 0 forces Python to check this folder FIRST before pip packages
    sys.path.insert(0, custom_src_path) 
    print(f"\nSUCCESS: Injected custom path at the top of sys.path:")
    print(f"-> {custom_src_path}")
else:
    print("\nPath is already registered in sys.path.")

=== 3. Dynamic Path Resolution ===
Current Directory: d:\PSEB Intership\MindGigs\Phase 2

SUCCESS: Injected custom path at the top of sys.path:
-> d:\PSEB Intership\MindGigs\src


## The Global PATH & OS Handoff

The Operating System does not inherently know what "Python" or "Docker" is. It only knows how to search. 

The `$PATH` variable is an ordered list of directories the OS scans when a command is executed in the terminal. If you encounter a "Command Not Found" error, the software is likely installed, but its directory is missing from this `$PATH` variable. 

Let's inspect what system commands are currently available to the subprocesses in this environment.

In [5]:
print("=== 4. OS-Level $PATH Inspection ===")

# Retrieve the PATH string and split it by the OS-specific separator (':' on Mac/Linux, ';' on Windows)
system_path_string = os.environ.get('PATH', '')
system_paths = system_path_string.split(os.pathsep)

print(f"The Operating System is currently searching {len(system_paths)} directories for executables.\n")
print("Top 5 search directories in priority order:")
for i, p in enumerate(system_paths[:5], 1):
    print(f" {i}. {p}")

=== 4. OS-Level $PATH Inspection ===
The Operating System is currently searching 18 directories for executables.

Top 5 search directories in priority order:
 1. c:\Users\zarya\anaconda3
 2. C:\Users\zarya\anaconda3
 3. C:\Users\zarya\anaconda3\Library\mingw-w64\bin
 4. C:\Users\zarya\anaconda3\Library\usr\bin
 5. C:\Users\zarya\anaconda3\Library\bin
